# 03 — Model: TF-IDF retriever + templated answer generator

**Project:** AI Career Copilot (H09)

The v0.1 retriever is a TF-IDF + cosine-similarity index over the job-posting corpus, packaged in `microcert_rec.models.fit_retriever` (in `career_copilot.models`). Production swaps in a multilingual embedding and a LangGraph agent loop with tool calls — the *interface* (a `posting_text` string corpus + a query) does not change.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity

sys.path.insert(0, str(Path.cwd().parent / 'src'))
from career_copilot.data import PROCESSED, load_all, make_training_artifacts
from career_copilot import models
from career_copilot.models import fit_retriever, save
from career_copilot.features import (
    posting_text, profile_text, blend_query_with_profile, skill_set, required_skill_set,
)

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 110

In [ ]:
if (PROCESSED / 'postings.parquet').exists():
    postings, profiles = load_all()
else:
    postings, profiles = make_training_artifacts()

## 1. Fit the retriever

In [ ]:
art = fit_retriever(postings)
print('vocab:', len(art['vec'].vocabulary_))
print('X shape:', art['X'].shape)

## 2. Top-k retrieval for a sample profile

In [ ]:
prof = profiles.iloc[0].to_dict()
query = 'I want to grow into a leadership role within data'
blended = blend_query_with_profile(query, prof)
qv = art['vec'].transform([blended])
sims = cosine_similarity(qv, art['X']).ravel()
top = np.argsort(-sims)[:5]
postings.iloc[top][['job_id', 'title', 'dept', 'level']].assign(sim=sims[top].round(3))

## 3. Templated answer generator

We assemble a templated answer: top-3 postings + a missing-skills callout. This is the v0.1 stand-in for the LLM-generated narrative.

In [ ]:
def render_answer(profile, postings_df, top_indices, sims):
    have = skill_set(profile.get('skills', ''))
    parts = []
    parts.append(f"Based on your current role ({profile.get('current_role','?')}, dept {profile.get('current_dept','?')}) and aspiration ({profile.get('aspiration','?')}), I found these postings:")
    for i in top_indices[:3]:
        row = postings_df.iloc[i]
        need = required_skill_set(row['required_skills'])
        missing = sorted(need - have)
        parts.append(f"- {row['title']} (L{row['level']}, {row['dept']}, sim={sims[i]:.2f}). Skills you'd need to add: {', '.join(missing[:5]) or '—'}.")
    parts.append('Decision aid only — talk to your manager before applying.')
    return '\n'.join(parts)

print(render_answer(prof, postings, top, sims))

## 4. Score-distribution chart for the retrieval

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.4))
ax.hist(sims, bins=40, color='#3a7ca5', edgecolor='white')
ax.set_xlabel('cosine similarity to retrieval query')
ax.set_title('Per-posting similarity for the sample query')
plt.tight_layout(); plt.show()

## 5. Per-dept retrieval coverage

Are some departments systematically retrieved more often? An asymmetry reveals an indexing or vocabulary bias we have to surface in the model card.

In [ ]:
rng = np.random.default_rng(0)
sample_p = profiles.sample(200, random_state=0)
dept_in_top5 = []
for _, p in sample_p.iterrows():
    qv = art['vec'].transform([blend_query_with_profile('grow my career', p.to_dict())])
    s = cosine_similarity(qv, art['X']).ravel()
    for i in np.argsort(-s)[:5]:
        dept_in_top5.append(postings.iloc[i]['dept'])
share = pd.Series(dept_in_top5).value_counts(normalize=True)
fig, ax = plt.subplots(figsize=(8, 3.4))
share.sort_values().plot(kind='bar', ax=ax, color='#9c6644')
ax.set_title('Top-5 department share across 200 sample queries')
ax.tick_params(axis='x', rotation=20)
plt.tight_layout(); plt.show()

## 6. Persist the retrieval index

In [ ]:
save(art, 'rag_index.joblib')
print('saved rag_index.joblib')

## 7. Hand-off

The retrieval index is now what `serve.chat` loads at startup. `04_eval.ipynb` exercises top-k coverage of the synthetic ground-truth `target_job_id` and the cross-dept slice, plus latency.